In [3]:
import numpy as np
import torch
import time

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Build the Modulo-2 Target Tensor
def build_target_f2():
    T = torch.zeros((9,9,9), dtype=torch.int8, device=DEVICE)
    for i in range(3):
        for j in range(3):
            for k in range(3):
                T[i*3+j, j*3+k, i*3+k] = 1
    return T

T_target = build_target_f2()

# 2. Modulo-2 Matrix Multiplication (No Einsum, just bitwise logic)
def calc_f2_tensor(U, V, W):
    # Using float temporarily just for the batch outer product, then modulo 2
    recon = torch.einsum("ir,jr,kr->ijk", U.float(), V.float(), W.float())
    return (recon.to(torch.int8) % 2)

# 3. Discrete Local Search (The Bit-Flipper)
def test_rank22_f2(U_full, V_full, W_full, col_drop, iterations=50000):
    start_t = time.time()
    keep = [i for i in range(23) if i != col_drop]
    
    # Isolate the remaining 22 columns
    U = U_full[:, keep].clone()
    
    V = V_full[:, keep].clone()
    W = W_full[:, keep].clone()
    
    # Calculate Initial Error (Hamming Distance)
    current_recon = calc_f2_tensor(U, V, W)
    current_error = (current_recon != T_target).sum().item()
    best_error = current_error
    
    print(f"Starting F2 Local Search | Initial Bit Errors: {current_error}/729")
    
    for i in range(iterations):
        # Pick a random matrix (U, V, or W), a random row, and a random column
        mat_choice = torch.randint(0, 3, (1,)).item()
        row = torch.randint(0, 9, (1,)).item()
        col = torch.randint(0, 22, (1,)).item()
        
        # Flip the bit (0 -> 1 or 1 -> 0)
        if mat_choice == 0: U[row, col] = 1 - U[row, col]
        elif mat_choice == 1: V[row, col] = 1 - V[row, col]
        else: W[row, col] = 1 - W[row, col]
            
        # Check new error
        new_recon = calc_f2_tensor(U, V, W)
        new_error = (new_recon != T_target).sum().item()
        
        # Greedy acceptance: If the flip helped (or kept it equal to allow walking flat areas), keep it
        if new_error <= current_error:
            current_error = new_error
            if new_error < best_error:
                best_error = new_error
                print(f"  Step {i:6d} | New Best F2 Error: {best_error}/729 mismatched bits")
                
            if best_error == 0:
                print(f"\n*** EXACT RANK-22 ROOT FOUND IN F2 AT STEP {i}! ***")
                break
        else:
            # Revert the bit flip if it made things worse
            if mat_choice == 0: U[row, col] = 1 - U[row, col]
            elif mat_choice == 1: V[row, col] = 1 - V[row, col]
            else: W[row, col] = 1 - W[row, col]
            
    print(f"Done in {time.time()-start_t:.1f}s | Final Best Error: {best_error}/729")
    return best_error

# --- EXECUTION ---
# Load your F2 schemes (using your np.load from the image)

d = np.load('factorizations_f2.npz', allow_pickle=True)

# EXPLICITLY load the 3x3x3 scheme
scheme = d['3,3,3']

# Safely extract if wrapped in a 0-d object array
if isinstance(scheme, np.ndarray) and scheme.ndim == 0:
    scheme = scheme.item()

# Route based on whether it's a dict or a pure array/list
if isinstance(scheme, dict):
    U_f2 = torch.tensor(scheme['u'], dtype=torch.int8, device=DEVICE)
    V_f2 = torch.tensor(scheme['v'], dtype=torch.int8, device=DEVICE)
    W_f2 = torch.tensor(scheme['w'], dtype=torch.int8, device=DEVICE)
else:
    # It's a standard array of shape (3, 9, 23)
    U_f2 = torch.tensor(scheme[0], dtype=torch.int8, device=DEVICE)
    V_f2 = torch.tensor(scheme[1], dtype=torch.int8, device=DEVICE)
    W_f2 = torch.tensor(scheme[2], dtype=torch.int8, device=DEVICE)

print(f"Successfully loaded 3x3x3 scheme with shape: {U_f2.shape}")

# Try dropping Column 0 to test
test_rank22_f2(U_f2, V_f2, W_f2, col_drop=0)

Successfully loaded 3x3x3 scheme with shape: torch.Size([9, 23])
Starting F2 Local Search | Initial Bit Errors: 46/729
  Step     32 | New Best F2 Error: 44/729 mismatched bits
  Step     78 | New Best F2 Error: 43/729 mismatched bits
  Step    101 | New Best F2 Error: 42/729 mismatched bits
  Step    142 | New Best F2 Error: 41/729 mismatched bits
  Step    148 | New Best F2 Error: 39/729 mismatched bits
  Step    163 | New Best F2 Error: 38/729 mismatched bits
  Step    201 | New Best F2 Error: 37/729 mismatched bits
  Step    331 | New Best F2 Error: 36/729 mismatched bits
  Step    338 | New Best F2 Error: 35/729 mismatched bits
  Step    475 | New Best F2 Error: 33/729 mismatched bits
  Step    651 | New Best F2 Error: 32/729 mismatched bits
  Step   1105 | New Best F2 Error: 31/729 mismatched bits
  Step   1119 | New Best F2 Error: 30/729 mismatched bits
  Step   1538 | New Best F2 Error: 29/729 mismatched bits
  Step   1811 | New Best F2 Error: 28/729 mismatched bits
  Step   25

27